# Reproduce paper figures and key tables

This notebook reproduces the manuscript-level figures and key tables for **Low-dimensional cortical geometry constrains linguistic representations**.

It is designed as a reproducibility dashboard rather than a full analysis notebook. By default, it displays already-generated files from `pang_out/`. Set `RUN_SCRIPTS = True` below to regenerate figures from the corresponding scripts in `code/`.

Expected repository structure:

```text
~/eigenmode_fingerprints/
  code/
  pang_out/
    paper_figures/
    paper_tables/
    word_transition_geometry/
```

The notebook does not rerun the full fMRI/eigenmode pipeline. For full re-execution, use the project pipeline scripts / `run_pipeline.sh`.


## 0. Setup


In [ ]:
from pathlib import Path
import subprocess
import sys
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

BASE = Path.home() / "eigenmode_fingerprints"
PANG = BASE / "pang_out"
FIG_DIR = PANG / "paper_figures"
TAB_DIR = PANG / "paper_tables"
CODE_DIR = BASE / "code"
WORD_GEOM_DIR = PANG / "word_transition_geometry"

RUN_SCRIPTS = False  # set True to regenerate figures/tables from scripts where available

print("BASE:", BASE)
print("PANG:", PANG)
print("FIG_DIR:", FIG_DIR)
print("TAB_DIR:", TAB_DIR)
print("CODE_DIR:", CODE_DIR)


In [ ]:
def run_script(script_name, *args, allow_missing=True):
    """Run a project script from code/ if RUN_SCRIPTS=True."""
    script = CODE_DIR / script_name
    if not RUN_SCRIPTS:
        print(f"[dry-run] python {script} {' '.join(args)}")
        return
    if not script.exists():
        msg = f"Missing script: {script}"
        if allow_missing:
            print("[skip]", msg)
            return
        raise FileNotFoundError(msg)
    cmd = [sys.executable, str(script), *map(str, args)]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)


def show_figure(filename_or_path, figsize=(10, 6), title=None):
    """Display a PNG/JPG figure from paper_figures/ or an explicit path."""
    path = Path(filename_or_path)
    if not path.is_absolute():
        path = FIG_DIR / path
    if not path.exists():
        print(f"[missing] {path}")
        return None
    img = mpimg.imread(path)
    plt.figure(figsize=figsize)
    plt.imshow(img)
    plt.axis("off")
    plt.title(title or path.name)
    plt.show()
    return path


def show_first_existing(candidates, figsize=(10, 6), title=None):
    """Display the first existing file from a list of candidate names/paths."""
    for c in candidates:
        path = Path(c)
        if not path.is_absolute():
            path = FIG_DIR / path
        if path.exists():
            return show_figure(path, figsize=figsize, title=title or path.name)
    print("[missing all candidates]")
    for c in candidates:
        print("  -", c)
    return None


def show_table(filename_or_path, n=20):
    path = Path(filename_or_path)
    if not path.is_absolute():
        # Try paper_tables first, then paper_figures for older outputs.
        candidate = TAB_DIR / path
        if not candidate.exists():
            candidate = FIG_DIR / path
        path = candidate
    if not path.exists():
        print(f"[missing] {path}")
        return None
    df = pd.read_csv(path)
    display(df.head(n))
    print("Rows:", len(df), "Columns:", list(df.columns))
    return df


## 1. Figure 1: scale-free eigenmode energy spectrum

This section displays the group-level eigenmode energy spectrum and the robustness / cumulative-energy summaries used to motivate the eigenmode description of language-related cortical activity.


In [ ]:
# Replace the script name here if your local figure-generation script differs.
run_script("plot_group_energy_spectrum.py")

show_first_existing([
    "figure_energy_spectrum.png",
    "figure_scale_free_energy_spectrum.png",
    "group_energy_fit_loglog_subjects.png",
], figsize=(9, 6), title="Figure 1: scale-free eigenmode energy spectrum")


In [ ]:
# Supplementary cumulative variance / low-dimensionality figure and table
show_first_existing([
    "figure_cumulative_eigenmode_energy.png",
    "figure_cumulative_variance_eigenmodes.png",
], figsize=(7, 5), title="Supplement: cumulative eigenmode variance")

show_table("table_cumulative_eigenmode_energy_group.csv")


## 2. Figure 2: sentence-level eigenmode response profiles

Boundary and sentence-shift regressors are compared in eigenmode space. The key claim is profile similarity despite different regressor definitions and magnitudes.


In [ ]:
run_script("plot_sentence_level_profiles.py")

show_first_existing([
    "figure_sentence_level_profiles.png",
    "figure_sentence_boundary_shift_profiles.png",
    "figure_sentence_level_eigenmode_profiles.png",
], figsize=(9, 5), title="Figure 2: sentence-level eigenmode response profiles")

show_table("table_sentence_level_summary.csv")


## 3. Figure 3: empirical reconstruction of sentence-level β maps

This is the main vertexwise validation figure: empirical sentence-boundary and sentence-shift β maps are reconstructed using the full retained eigenmode basis and the first 20 non-constant modes.


In [ ]:
run_script("figure_sentence_level_empirical_reconstruction_twopanel.py")

show_figure("figure_sentence_level_empirical_reconstruction_twopanel_K20.png", figsize=(14, 9),
            title="Figure 3: sentence-level empirical β map reconstruction")

show_table("figure_sentence_level_empirical_reconstruction_twopanel_K20_stats.csv")


## 4. Figure 4: token-level transition metrics

This figure summarizes boundary-related increases in token-level transition metrics and their Pearson correlation structure in representational space. Error bars in Panel A indicate SEM.


In [ ]:
run_script("plot_linguistic_composite_AB.py")

show_figure("figure_transition_metrics_boundary_and_correlations.png", figsize=(11, 5),
            title="Figure 4: transition metrics and Pearson correlations")

# Raw summary tables used by the figure
show_table(WORD_GEOM_DIR / "word_transition_summary.csv")


## 5. Figure 5: low-order eigenmodes and low-pass reconstructed regressor maps

This overview figure visualizes the low-order cortical eigenmodes and the maps reconstructed from the first 20 eigenmode β coefficients for all six linguistic regressors.


In [ ]:
run_script("figure_modes_plus_beta_pial.py")

show_figure("figure_modes_plus_beta_pial.png", figsize=(12, 12),
            title="Figure 5: low-order eigenmodes and reconstructed regressor maps")


## 6. Figure 6: PCA of eigenmode β profiles

PCA summarizes profile convergence across regressors. The first component captures the dominant long-wavelength eigenmode profile shared across linguistic metrics.


In [ ]:
run_script("plot_pca_eigenmode_profiles.py")

show_first_existing([
    "figure_pca_eigenmode_profiles.png",
    "figure_pca_beta_profiles.png",
    "figure_pca_profile_collapse.png",
], figsize=(10, 6), title="Figure 6: PCA of eigenmode β profiles")


## 7. Supplementary: reconstruction performance across all regressors

This table summarizes reconstruction quality for full retained modes and for the K=20 low-pass reconstruction.


In [ ]:
run_script("make_empirical_reconstruction_summary_table.py")
show_table("table_empirical_lowpass_reconstruction_summary.csv")


In [ ]:
# Optional: display individual empirical/full/low-pass/residual reconstruction figures for all regressors.
for metric in ["boundary", "sentence_shift", "token_shift", "pred_error_ar", "pred_error_subspace", "curvature"]:
    if RUN_SCRIPTS:
        run_script("figure_empirical_full_lowpass_residual.py", "--metric", metric, "--k-low", "20")
    show_first_existing([
        f"figure_empirical_full_lowpass_residual_{metric}_K20.png",
        f"figure_empirical_reconstruction_{metric}_K20.png",
    ], figsize=(11, 8), title=f"Supplement: empirical reconstruction — {metric}")


## 8. Supplementary: correlations among low-pass reconstructed maps

These correlations quantify the sentence-level versus token-level clustering seen visually in the reconstructed maps.


In [ ]:
run_script("analyze_reconstructed_map_correlations.py")

show_first_existing([
    "figure_reconstructed_map_correlations.png",
], figsize=(6, 5), title="Supplement: low-pass reconstructed map correlations")

# This CSV was saved to paper_figures in the original script; also check paper_tables in case you moved it.
show_table("table_reconstructed_map_correlations.csv")


## 9. Supplementary: word-rate and content-density control

Boundary eigenmode profiles are compared before and after controlling for word rate and content-word density.


In [ ]:
show_first_existing([
    "figure_boundary_wordrate_content_control.png",
    "figure_boundary_control_wordrate_content.png",
    "figure_sentence_boundary_controls.png",
], figsize=(9, 5), title="Supplement: boundary controls")

show_table("group_boundary_wordrate_content_by_mode_subject_level.csv")


## 10. Supplementary: residualized joint GLMs

These analyses test whether prediction error, subspace exit, and curvature retain unique variance beyond token shift, and whether that unique variance preserves the same eigenmode profile shape.


In [ ]:
show_first_existing([
    "figure3_residualized_joint_glms.png",
    "figure_residualized_joint_glms.png",
], figsize=(12, 5), title="Supplement: residualized joint GLMs")

show_table("table_residualized_joint_glm_raw_amplitudes.csv")


## 11. Supplementary: curvature-family convergence

Multiple geometric trajectory descriptors are compared to test whether the curvature result depends on a specific operationalization.


In [ ]:
show_first_existing([
    "figure_curvature_family_collapse_zscore.png",
    "figure_curvature_family_collapse.png",
    "figure_curvature_only.png",
], figsize=(10, 5), title="Supplement: curvature-family convergence")


## 12. Supplementary: Mode 3 alignment with canonical cortical maps

Mode 3 is correlated with canonical functional, microstructural, and morphological maps.


In [ ]:
show_table("table_mode3_canonical_map_correlations_clean.csv")

show_first_existing([
    "figure_mode3_neuromaps_correlations.png",
    "figure_mode3_canonical_map_correlations.png",
], figsize=(8, 5), title="Supplement: Mode 3 canonical-map correlations")


## 13. Supplementary: embedding-model robustness

Token-level transition metrics and eigenmode β profiles are compared across BERT and Qwen3 embeddings.


In [ ]:
show_first_existing([
    "figure_qwen_boundary_transition_metrics.png",
    "figure_bert_qwen_transition_metrics.png",
], figsize=(10, 5), title="Supplement: BERT/Qwen transition metrics")

show_first_existing([
    "figure_qwen_bert_eigenmode_profiles.png",
    "figure_embedding_model_robustness_profiles.png",
], figsize=(10, 6), title="Supplement: BERT/Qwen eigenmode profile robustness")

show_table("table_embedding_model_profile_similarity.csv")


## 14. Subcortical hippocampus and VTA analyses

The manuscript additionally includes hippocampal and brainstem analyses implemented in `code/subcortex/`. These scripts extend the cortical eigenmode framework to hippocampal graph eigenmodes, VTA/LC ROI analyses, VTA–hippocampal coupling, and VTA–cortical eigenmode coupling.

Representative figure-generation scripts:

- `code/subcortex/plot_hippocampus_all_results_summary.py`
- `code/subcortex/plot_main_subcortical_updating_network.py`
- `code/subcortex/plot_main_vta_cortical_overlap.py`
- `code/subcortex/plot_vta_hipp_coupling_global_residualized.py`
- `code/subcortex/plot_supp_hipp_trajectory_control.py`

These scripts assume that the corresponding precomputed outputs exist under `pang_out/subcortex/`.


In [ ]:
from pathlib import Path

subcortex_dir = REPO / 'code' / 'subcortex'
subcortex_outputs = REPO / 'pang_out' / 'subcortex'

print('Subcortex code directory exists:', subcortex_dir.exists())
print('Subcortex output directory exists:', subcortex_outputs.exists())

figure_scripts = [
    'plot_hippocampus_all_results_summary.py',
    'plot_main_subcortical_updating_network.py',
    'plot_main_vta_cortical_overlap.py',
    'plot_vta_hipp_coupling_global_residualized.py',
    'plot_supp_hipp_trajectory_control.py',
]

for script in figure_scripts:
    print(f'{script}:', (subcortex_dir / script).exists())


## 15. Output checklist

This cell checks whether the key manuscript outputs are present. Missing files may simply indicate that local filenames differ; update the candidate names above if needed.


In [ ]:
expected_figures = [
    "figure_energy_spectrum.png",
    "figure_sentence_level_empirical_reconstruction_twopanel_K20.png",
    "figure_transition_metrics_boundary_and_correlations.png",
    "figure_modes_plus_beta_pial.png",
]

expected_tables = [
    "figure_sentence_level_empirical_reconstruction_twopanel_K20_stats.csv",
    "table_empirical_lowpass_reconstruction_summary.csv",
    "table_mode3_canonical_map_correlations_clean.csv",
]

print("Figures:")
for f in expected_figures:
    p = FIG_DIR / f
    print(("OK     " if p.exists() else "MISSING"), p)

print("\nTables:")
for f in expected_tables:
    p = TAB_DIR / f
    print(("OK     " if p.exists() else "MISSING"), p)


## 16. Environment information


In [ ]:
print("Python:", sys.version)
print("Executable:", sys.executable)
print("Platform:", platform.platform())

for mod_name in ["numpy", "pandas", "matplotlib", "scipy", "nilearn", "nibabel"]:
    try:
        mod = __import__(mod_name)
        print(f"{mod_name}:", getattr(mod, "__version__", "unknown"))
    except Exception as e:
        print(f"{mod_name}: not importable ({e})")
